# Playground Series S6E8: Predicting Smartphone Addiction — S6E8 | U Smart Phone Addict? 解説付き写し

- **コンペ**: [Predicting Smartphone Addiction (Playground Series S6E8)](https://www.kaggle.com/competitions/playground-series-s6e8)
- **元notebook**: [S6E8 | U Smart Phone Addict?](https://www.kaggle.com/code/anhadmahajan06/s6e8-u-smart-phone-addict)（ノートブック内タイトルは「S6E8: Continuous Blender (Future-Proof)」）by Anhad Mahajan
- **スコア**: Public Score 0.97092（コード一覧のBest Score表示）
- **手法の概要**: モデルそのものを訓練するのではなく、Kaggleデータセットにアップロードされた「スコア付きの複数の提出済みCSVファイル」をファイル名から自動検出し、順位（ランク）ベースで5種類の異なるアンサンブル（Linear Anchor, Power Rank Decay, Top3 Average, Geometric Mean, Sharp Power Blend）を動的に生成する「継続的ブレンダー」notebook。

**注記**: これは学習目的の解説付き写しです。元notebookのコードは、HTMLからのテキスト抽出時にインデント（字下げ）情報が失われていたため、ロジック・トークンは変えずにPythonとして妥当な形にインデントを復元しています（未実行、出力は含みません）。

## 評価指標（簡潔）

Playground Seriesの本コンペは**ROC-AUC**（予測確率のランキングの質を測る指標。0.5がランダム、1.0が完全な分離）。ROC-AUCは確率の絶対値ではなく順位（ランク）だけに依存するため、このnotebookのように複数モデルの予測を「順位（rank）」に変換してから平均するアンサンブル手法は、この指標と相性が良い。実際、以前のバージョンで確率をロジット変換（無限大に発散しうる変換）してブレンドしたところスコアが下がった経緯が説明されており、「順位を0〜1の範囲に収めたまま混ぜる」設計に切り替えたと明記されている。

## 1. 依存ライブラリの読み込み

**What**: ランキング処理用の`scipy.stats.rankdata`（順位変換）と`gmean`（幾何平均）、ファイル探索用の`pathlib.Path`などをインポートする。

**Why**: このnotebookの核心は「複数の提出ファイルを順位に変換してブレンドする」ことなので、`rankdata`（値を1〜Nの順位に変換する関数）と`gmean`（幾何平均、後述のROC-AUCとの相性の良さに関わる）が中心的な役割を果たす。初心者向け補足: `rankdata`は例えば`[0.3, 0.9, 0.1]`を`[2, 3, 1]`のような順位に変換する関数で、外れ値の影響を受けにくくする効果がある。

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.stats import gmean
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
print("Dependencies Loaded.")


## 2. スコア付き提出ファイルの自動検出と読み込み

**What**: `/kaggle/input`配下を再帰的に走査し、ファイル名に`0.xxxxx`のようなスコアらしき数値パターンが含まれるCSVファイルをすべて検出する。各ファイルからスコアを正規表現で抽出し、スコアをキーとした辞書`subs_dict`に予測値配列を格納、スコアの高い順にソートする。

**Why**: 「後から新しい提出ファイルをKaggleデータセットに追加するだけで、コードを一切変更せずにアンサンブルに組み込める」という設計思想（notebookタイトルの"Future-Proof"＝将来のモデル追加に対応）。正規表現`r"0\.[0-9]{4,}"`でファイル名からスコアを抽出することで、手作業でのファイルリスト管理を不要にしている。初心者向け補足: `Path.rglob(pattern)`はディレクトリを再帰的に探索してパターンに一致するファイルを全て見つけるメソッドで、フォルダ構造を意識せず一括探索したいときに便利。

In [ ]:
# Find Sample Submission
search_dirs = [Path('/kaggle/input'), Path('/input'), Path('.'), Path('..')]
sample_sub_path = Path('./playground-series-s6e8/sample_submission.csv')
for base in search_dirs:
    if not base.exists():
        continue
    for p in base.rglob('sample_submission.csv'):
        if 'playground-series' in str(p).lower():
            sample_sub_path = p

# Search ALL directories for scored CSVs
search_base = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
sub_files = []
for p in search_base.rglob('*.csv'):
    if re.search(r"0\.[0-9]{4,}", p.name):
        sub_files.append(p)

print(f"Found {len(sub_files)} scored submission files.")

# Load unique models
subs_dict = {}
for f in sub_files:
    match = re.search(r"0\.[0-9]{4,}", f.name)
    score = float(match.group(0)) if match else 0.0

    if score > 0 and score not in subs_dict:
        df = pd.read_csv(f).sort_values('id').reset_index(drop=True)
        subs_dict[score] = df['addicted_label'].values

# Sort dict from highest score to lowest
subs_dict = dict(sorted(subs_dict.items(), reverse=True))
scores = list(subs_dict.keys())
N = len(subs_dict[scores[0]])

print(f"Loaded {len(scores)} unique models.")
print(f"  Top Anchor Model: {scores[0]:.5f}")


## 3. 5種類のランクベースアンサンブルを生成

**What**: `save_sub`ヘルパー関数を定義したうえで、(0)最高スコアモデルそのまま、(1)最高スコアモデルに95%の重みを与えた「Linear Anchor」、(2)スコア順位に応じて指数的に重みを減衰させる「Power Rank Decay」、(3)上位3モデルの単純平均「Top3 Linear Average」、(4)上位5モデルの幾何平均「Geometric Mean」、(5)順位を2乗して差を強調する「Sharp Power Blend」、という5つの異なるアンサンブル戦略でそれぞれ提出用CSVを書き出す。

**Why**: 単一のブレンド手法に頼らず、性質の異なる複数のアンサンブル（線形 vs 指数減衰 vs 幾何平均 vs べき乗強調）を並行して用意しておくことで、実際のリーダーボードでどれが最も安定して良いスコアを出すかを試行錯誤できるようにしている。特に幾何平均は「どれか1モデルが極端に低い確率（0に近い）を出すと全体が強く下がる」性質があり、ROC-AUCでは1つのモデルが自信満々に間違えるケースへの罰則として働きやすい。初心者向け補足: 「アンサンブル」とは複数モデルの予測を組み合わせて1つの予測にする手法の総称で、単独モデルより安定して高い性能を出しやすいことが多い。

In [ ]:
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
sample_df = pd.read_csv(sample_sub_path).sort_values('id').reset_index(drop=True)


def save_sub(preds, filename):
    sub = sample_df.copy()
    sub['addicted_label'] = preds
    filepath = OUTPUT_DIR / filename
    sub.to_csv(filepath, index=False)
    print(f"  Saved: {filename}")


# ---------------------------------------------------------
# 0. Raw Top Model (Baseline Safety)
# Simply outputs the highest-scoring dataset file exactly as-is.
# ---------------------------------------------------------
save_sub(subs_dict[scores[0]], 'submission.csv')

# ---------------------------------------------------------
# 1. Linear Extreme Anchor (Safe & Bounded)
# No logit infinities. Just pure 95% Top Model + 5% Rest.
# ---------------------------------------------------------
linear_anchor = (rankdata(subs_dict[scores[0]]) / N) * 0.95
support_weight = 0.05 / (len(scores) - 1) if len(scores) > 1 else 0.0
for i in range(1, len(scores)):
    linear_anchor += (rankdata(subs_dict[scores[i]]) / N) * support_weight
save_sub(linear_anchor, '1_linear_anchor.csv')

# ---------------------------------------------------------
# 2. Power Rank Decay (Top models get massive weight)
# ---------------------------------------------------------
power_decay = np.zeros(N)
weights = [0.5 ** i for i in range(len(scores))]
weights = [w / sum(weights) for w in weights]
for i in range(len(scores)):
    power_decay += (rankdata(subs_dict[scores[i]]) / N) * weights[i]
save_sub(power_decay, '2_power_rank_decay.csv')

# ---------------------------------------------------------
# 3. Top 3 Linear Average
# ---------------------------------------------------------
top3_avg = np.zeros(N)
top_k = min(3, len(scores))
for i in range(top_k):
    top3_avg += (rankdata(subs_dict[scores[i]]) / N) / top_k
save_sub(top3_avg, '3_top3_linear_avg.csv')

# ---------------------------------------------------------
# 4. Geometric Mean of Top 5
# Excellent for ROC-AUC as it severely punishes models that predict 0.
# ---------------------------------------------------------
top_k_geom = min(5, len(scores))
geom_matrix = np.zeros((N, top_k_geom))
for i in range(top_k_geom):
    geom_matrix[:, i] = np.clip(subs_dict[scores[i]], 1e-6, 1 - 1e-6)
geom_blend = gmean(geom_matrix, axis=1)
save_sub(geom_blend, '4_geom_mean_top5.csv')

# ---------------------------------------------------------
# 5. Non-Linear Power Sharpness (Exponent = 2.0)
# Pushes confident predictions even higher, stretches the middle.
# ---------------------------------------------------------
sharp_blend = np.zeros(N)
for i in range(len(scores)):
    sharp_blend += ((rankdata(subs_dict[scores[i]]) / N) ** 2.0) * weights[i]
save_sub(sharp_blend, '5_sharp_power_blend.csv')

print("\nContinuous Blender Complete! Submit `1_linear_anchor.csv` first.")
